In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType



In [0]:
df = spark.table('workspace.bronze.erp_loc_a101')

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, F.trim(F.col(field.name)))

In [0]:
df.limit(10).display()

In [0]:

df = df.withColumn('CNTRY',
                   F.when(F.col('CNTRY').isin('US', 'USA'), 'United States')
                   .when(F.col('CNTRY').isin('DE'), 'Germany')
                   .when(F.col('CNTRY').isNull() | (F.col('CNTRY') == ""), 'n/a')
                   .otherwise(F.col('CNTRY'))
                   )

In [0]:
df = df.withColumn("cid", F.regexp_replace(F.col("cid"), "-", ""))

In [0]:
RENAME_MAP = {
    "CID": "customer_key",
    "CNTRY": "country"
}

for key, value in RENAME_MAP.items():
    df = df.withColumnRenamed(key,value)

In [0]:
df.limit(10).display()

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_customer_location")

In [0]:
%sql

SELECT *
FROM workspace.silver.erp_customer_location
LIMIT 100;